In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
!pip install mlflow dagshub -q
import warnings
warnings.filterwarnings('ignore')
os.environ["MLFLOW_TRACKING_USERNAME"] = "giomamaca"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "095241f97fd50006565a872843e1054bb2ae82cb"

import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectFromModel


import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import mlflow
import mlflow.xgboost
import mlflow.sklearn

import matplotlib.pyplot as plt
import seaborn as sns

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 63.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 64.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
DAGSHUB_USER = 'giomamaca'
DAGSHUB_REPO = 'ML2'

mlflow.set_tracking_uri(f'https://dagshub.com/{DAGSHUB_USER}/{DAGSHUB_REPO}.mlflow')

EXPERIMENT_NAME = 'XGBoost_Training'
mlflow.set_experiment(EXPERIMENT_NAME)

SEED = 42
np.random.seed(SEED)

In [4]:
train_trans  = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_id     = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test_trans   = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_id      = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

train = train_trans.merge(train_id, on='TransactionID', how='left')
test  = test_trans.merge(test_id,  on='TransactionID', how='left')

print('Train shape:', train.shape)
print('Test shape :', test.shape)
print('Fraud rate  :', train['isFraud'].mean().round(4))

Train shape: (590540, 434)
Test shape : (506691, 433)
Fraud rate  : 0.035


## Preprocessing

In [5]:
class FraudPreprocessor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = X.copy()

        # Drop cols
        nan_frac        = X.isnull().mean()
        self.nan_drop_  = nan_frac[nan_frac > 0.90].index.tolist()
        num_cols        = [c for c in X.select_dtypes('number').columns
                           if c not in ['TransactionID', 'isFraud']]
        self.near_zero_ = X[num_cols].std()[X[num_cols].std() < 0.01].index.tolist()
        X.drop(columns=self.nan_drop_ + self.near_zero_, inplace=True, errors='ignore')

        # Add features
        X = self._add_features(X)

        # Frequency maps
        self.freq_maps_ = {}
        for col in ['card1', 'card2', 'addr1', 'P_emaildomain']:
            if col in X.columns:
                self.freq_maps_[col] = X[col].value_counts(normalize=True).to_dict()

        # Group stats
        self.grp_stats_ = {}
        for grp in ['card1', 'card2']:
            if grp in X.columns:
                stats = X.groupby(grp)['TransactionAmt'].agg(['mean', 'std', 'count'])
                stats.columns = [f'{grp}_amt_{s}' for s in ['mean', 'std', 'count']]
                self.grp_stats_[grp] = stats

        # Card freq
        if 'card1' in X.columns and 'card2' in X.columns:
            combined = X['card1'].astype(str) + '_' + X['card2'].astype(str)
            self.card_freq_ = combined.value_counts(normalize=True).to_dict()

        # Label encoders — ყველა object სვეტზე
        self.encoders_ = {}
        for col in X.select_dtypes('object').columns:
            if col not in ['TransactionID']:
                le = LabelEncoder()
                le.fit(X[col].astype(str).fillna('unknown'))
                self.encoders_[col] = le

        print(f'Label encoders fit on {len(self.encoders_)} columns: {list(self.encoders_.keys())}')

        # Learn feature names
        self.feature_names_ = self._transform(X.copy()).columns.tolist()
        return self

    def _add_features(self, X):
        # OS and browser
        if 'id_30' in X.columns:
            X['os'] = X['id_30'].apply(
                lambda v: next((o for o in ['android','ios','windows','mac','linux']
                                if o in str(v).lower()), 'other') if pd.notna(v) else 'unknown')
        if 'id_31' in X.columns:
            X['browser'] = X['id_31'].apply(
                lambda v: next((b for b in ['chrome','safari','firefox','edge','samsung','opera']
                                if b in str(v).lower()), 'other') if pd.notna(v) else 'unknown')

        # Time features
        if 'TransactionDT' in X.columns:
            X['transaction_day']     = (X['TransactionDT'] // 86400).astype('int32')
            X['transaction_hour']    = ((X['TransactionDT'] // 3600) % 24).astype('int32')
            X['transaction_weekday'] = (X['transaction_day'] % 7).astype('int32')
            X['transaction_month']   = (X['transaction_day'] // 30).astype('int32')
            X['is_night']            = X['transaction_hour'].between(0, 6).astype('int8')
            X['is_weekend']          = X['transaction_weekday'].isin([5, 6]).astype('int8')

        # Amount features
        X['TransactionAmt_log']   = np.log1p(X['TransactionAmt'])
        X['TransactionAmt_cents'] = (X['TransactionAmt'] * 100 % 100).astype('float32')
        X['amt_is_round']         = (X['TransactionAmt_cents'] == 0).astype('int8')

        # Email features
        if 'P_emaildomain' in X.columns:
            X['P_email_suffix'] = X['P_emaildomain'].apply(
                lambda x: str(x).split('.')[-1] if pd.notna(x) else 'unknown')
        if 'R_emaildomain' in X.columns:
            X['R_email_suffix'] = X['R_emaildomain'].apply(
                lambda x: str(x).split('.')[-1] if pd.notna(x) else 'unknown')
        if 'P_emaildomain' in X.columns and 'R_emaildomain' in X.columns:
            X['email_match'] = (X['P_emaildomain'] == X['R_emaildomain']).astype('int8')

        return X

    def _transform(self, X):
        # Frequency encoding
        for col, fmap in getattr(self, 'freq_maps_', {}).items():
            if col in X.columns:
                X[f'{col}_freq'] = X[col].map(fmap).fillna(0).astype('float32')

        # Group aggregations
        for grp, stats in getattr(self, 'grp_stats_', {}).items():
            if grp in X.columns:
                X = X.merge(stats, on=grp, how='left')

        # Card combination frequency
        if hasattr(self, 'card_freq_') and 'card1' in X.columns and 'card2' in X.columns:
            combined = X['card1'].astype(str) + '_' + X['card2'].astype(str)
            X['card1_card2_freq'] = combined.map(self.card_freq_).fillna(0).astype('float32')

        # Label encoding
        for col, le in getattr(self, 'encoders_', {}).items():
            if col in X.columns:
                X[col] = X[col].astype(str)
                known  = set(le.classes_)
                X[col] = X[col].apply(lambda v: v if v in known else le.classes_[0])
                X[col] = le.transform(X[col])

        # Drop non-feature columns
        drop = [c for c in ['TransactionID', 'TransactionDT', 'isFraud'] if c in X.columns]
        X.drop(columns=drop, inplace=True)

        return X.fillna(-999)

    def transform(self, X):
        X = X.copy()
        drop = [c for c in self.nan_drop_ + self.near_zero_ if c in X.columns]
        X.drop(columns=drop, inplace=True)
        for col in X.select_dtypes('float64').columns:
            X[col] = X[col].astype('float32')
        for col in X.select_dtypes('int64').columns:
            X[col] = X[col].astype('int32')
        X = self._add_features(X)
        X = self._transform(X)
    
        for col in self.feature_names_:
            if col not in X.columns:
                X[col] = -999
        return X[self.feature_names_].fillna(-999)

## Cleaning + Feature Engineering

In [6]:
preprocessor = FraudPreprocessor()

with mlflow.start_run(run_name='XGBoost_Cleaning'):
    preprocessor.fit(train.drop(columns=['isFraud']))
    mlflow.log_params({
        'dropped_high_nan' : len(preprocessor.nan_drop_),
        'dropped_near_zero': len(preprocessor.near_zero_),
        'remaining_cols'   : len(preprocessor.feature_names_)
    })
    print(f'Dropped NaN cols : {len(preprocessor.nan_drop_)}')
    print(f'Dropped near-zero: {len(preprocessor.near_zero_)}')

with mlflow.start_run(run_name='XGBoost_Feature_Engineering'):
    mlflow.log_metric('total_features', len(preprocessor.feature_names_))
    print(f'Total features: {len(preprocessor.feature_names_)}')

X      = preprocessor.transform(train.drop(columns=['isFraud']))
X_test = preprocessor.transform(test)
y      = train['isFraud']

print('X shape     :', X.shape)
print('X_test shape:', X_test.shape)

Label encoders fit on 33 columns: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo', 'os', 'browser', 'P_email_suffix', 'R_email_suffix']
Dropped NaN cols : 12
Dropped near-zero: 2
🏃 View run XGBoost_Cleaning at: https://dagshub.com/giomamaca/ML2.mlflow/#/experiments/0/runs/70770840125d47aa922ff4fc1916dda2
🧪 View experiment at: https://dagshub.com/giomamaca/ML2.mlflow/#/experiments/0
Total features: 442
🏃 View run XGBoost_Feature_Engineering at: https://dagshub.com/giomamaca/ML2.mlflow/#/experiments/0/runs/db0284bdb867409fb26221b86a731a5b
🧪 View experiment at: https://dagshub.com/giomamaca/ML2.mlflow/#/experiments/0
X shape     : (590540, 442)
X_test shape: (506691, 442)


## Feature Selection

In [7]:
TARGET = 'isFraud'
DROP_COLS = ['TransactionID', 'TransactionDT', TARGET]

all_features = [c for c in train.columns if c not in DROP_COLS]
X = preprocessor.transform(train.drop(columns=['isFraud']))
y = train[TARGET]

In [8]:
for col in X.select_dtypes('object').columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str).fillna('unknown'))

print(X.select_dtypes('object').columns.tolist())
print(X.shape)

[]
(590540, 442)


In [9]:
SKF  = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

BASE_PARAMS = dict(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='auc', use_label_encoder=False,
    random_state=SEED, n_jobs=-1, tree_method='hist'
)

with mlflow.start_run(run_name='XGBoost_Feature_Selection'):

    num_X   = X.select_dtypes(include='number')
    corr    = num_X.corr().abs()
    upper   = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    drop_corr = [col for col in upper.columns if any(upper[col] > 0.95)]
    feats_A = [c for c in X.columns if c not in drop_corr]
    score_A = cross_val_score(xgb.XGBClassifier(**BASE_PARAMS),
                              X[feats_A], y, cv=SKF, scoring='roc_auc').mean()

    xgb_imp = xgb.XGBClassifier(**BASE_PARAMS)
    xgb_imp.fit(X, y)
    importances = pd.Series(xgb_imp.feature_importances_, index=X.columns)
    feats_B     = importances.nlargest(150).index.tolist()
    score_B     = cross_val_score(xgb.XGBClassifier(**BASE_PARAMS),
                                  X[feats_B], y, cv=SKF, scoring='roc_auc').mean()

    feats_C = [f for f, keep in zip(X.columns,
               SelectFromModel(xgb_imp, threshold='median', prefit=True).get_support()) if keep]
    score_C = cross_val_score(xgb.XGBClassifier(**BASE_PARAMS),
                              X[feats_C], y, cv=SKF, scoring='roc_auc').mean()

    BEST_FS_NAME, BEST_FS_SCORE, SELECTED_FEATURES = max(
        [('A_corr', score_A, feats_A),
         ('B_top150', score_B, feats_B),
         ('C_sfm', score_C, feats_C)],
        key=lambda x: x[1]
    )

    mlflow.log_metrics({'score_A': score_A, 'score_B': score_B,
                        'score_C': score_C, 'best_score': BEST_FS_SCORE})
    mlflow.log_param('best_approach', BEST_FS_NAME)

🏃 View run XGBoost_Feature_Selection at: https://dagshub.com/giomamaca/ML2.mlflow/#/experiments/0/runs/d1d055d20f9b4a5298583d936767e937
🧪 View experiment at: https://dagshub.com/giomamaca/ML2.mlflow/#/experiments/0


## Training

In [10]:
X_sel  = X[SELECTED_FEATURES]
X_test = X_test[SELECTED_FEATURES]

X_tr, X_val, y_tr, y_val = train_test_split(
    X_sel, y, test_size=0.2, random_state=SEED, stratify=y
)

def get_state(train_auc, val_auc):
    gap = train_auc - val_auc
    if val_auc < 0.80 and gap < 0.02:
        return 'underfitting'
    elif gap > 0.05:
        return 'overfitting'
    else:
        return 'good'


In [16]:
def run_experiment(run_name, params, X_tr, y_tr, X_val, y_val, early_stop=False):
    with mlflow.start_run(run_name=run_name):
        if early_stop:
            m = xgb.XGBClassifier(**params, early_stopping_rounds=50)
            m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        else:
            m = xgb.XGBClassifier(**params)
            m.fit(X_tr, y_tr)
        train_auc = roc_auc_score(y_tr, m.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
        state     = get_state(train_auc, val_auc)
        print(f'{run_name}: Train={train_auc:.5f}  Val={val_auc:.5f}  → {state}')
        mlflow.log_params(params)
        mlflow.log_metrics({'train_auc': train_auc, 'val_auc': val_auc,
                            'gap': train_auc - val_auc})
        mlflow.log_param('model_state', state)
    return val_auc

In [12]:
run_experiment(
    run_name   = 'XGBoost_depth2_trees30',
    params     = dict(n_estimators=30, max_depth=2, learning_rate=0.3,
                      eval_metric='auc', use_label_encoder=False,
                      random_state=SEED, tree_method='hist'),
    X_tr=X_tr, y_tr=y_tr, X_val=X_val, y_val=y_val
)

XGBoost_depth2_trees30: Train=0.87602  Val=0.87269  → good
🏃 View run XGBoost_depth2_trees30 at: https://dagshub.com/giomamaca/ML2.mlflow/#/experiments/0/runs/883443aefec249e095d9bba7f9cd9e40
🧪 View experiment at: https://dagshub.com/giomamaca/ML2.mlflow/#/experiments/0


np.float64(0.872692904976222)

In [13]:
run_experiment(
    run_name   = 'XGBoost_depth12_noreg',
    params     = dict(n_estimators=1000, max_depth=12, learning_rate=0.3,
                      reg_alpha=0, reg_lambda=0,
                      eval_metric='auc', use_label_encoder=False,
                      random_state=SEED, tree_method='hist'),
    X_tr=X_tr, y_tr=y_tr, X_val=X_val, y_val=y_val
)

XGBoost_depth12_noreg: Train=1.00000  Val=0.97295  → good
🏃 View run XGBoost_depth12_noreg at: https://dagshub.com/giomamaca/ML2.mlflow/#/experiments/0/runs/c142ecd2066d40e9905985dc46877344
🧪 View experiment at: https://dagshub.com/giomamaca/ML2.mlflow/#/experiments/0


np.float64(0.9729541845291354)

In [17]:
run_experiment(
    run_name   = 'XGBoost_depth6_baseline',
    params     = dict(n_estimators=500, max_depth=6, learning_rate=0.05,
                      subsample=0.8, colsample_bytree=0.8,
                      reg_alpha=0, reg_lambda=1,
                      scale_pos_weight=(y==0).sum()/(y==1).sum(),
                      eval_metric='auc', use_label_encoder=False,
                      random_state=SEED, tree_method='hist'),
    X_tr=X_tr, y_tr=y_tr, X_val=X_val, y_val=y_val,
    early_stop=True
)

XGBoost_depth6_baseline: Train=0.97589  Val=0.95439  → good
🏃 View run XGBoost_depth6_baseline at: https://dagshub.com/giomamaca/ML2.mlflow/#/experiments/0/runs/b248f6153e2343d58c904971240e507d
🧪 View experiment at: https://dagshub.com/giomamaca/ML2.mlflow/#/experiments/0


np.float64(0.954389426115547)

In [20]:
with mlflow.start_run(run_name='XGBoost_depth6_baseline'):
    params = dict(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0, reg_lambda=1,
        scale_pos_weight=(y==0).sum()/(y==1).sum(),
        eval_metric='auc', use_label_encoder=False,
        random_state=SEED, tree_method='hist',
        early_stopping_rounds=50          # ← moved here
    )
    m = xgb.XGBClassifier(**params)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    train_auc = roc_auc_score(y_tr, m.predict_proba(X_tr)[:, 1])
    val_auc   = roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
    state     = get_state(train_auc, val_auc)
    print(f'Train: {train_auc:.5f}  Val: {val_auc:.5f}  → {state}')
    mlflow.log_params(params)
    mlflow.log_metrics({'train_auc': train_auc, 'val_auc': val_auc, 'gap': train_auc - val_auc})
    mlflow.log_param('model_state', state)

Train: 0.97589  Val: 0.95439  → good
🏃 View run XGBoost_depth6_baseline at: https://dagshub.com/giomamaca/ML2.mlflow/#/experiments/0/runs/cdc89cc6dcbe417fbb83aa3f841dc897
🧪 View experiment at: https://dagshub.com/giomamaca/ML2.mlflow/#/experiments/0


In [22]:
def objective(trial):
    params = dict(
        n_estimators       = trial.suggest_int('n_estimators', 200, 1000, step=100),
        max_depth          = trial.suggest_int('max_depth', 3, 10),
        learning_rate      = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        subsample          = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree   = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        gamma              = trial.suggest_float('gamma', 0, 5),
        reg_alpha          = trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        reg_lambda         = trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        min_child_weight   = trial.suggest_int('min_child_weight', 1, 10),
        scale_pos_weight   = (y==0).sum()/(y==1).sum(),
        eval_metric        = 'auc',
        early_stopping_rounds = 30,
        random_state       = SEED,
        n_jobs             = -1,
        tree_method        = 'hist'
    )
    m = xgb.XGBClassifier(**params)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    return roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])

with mlflow.start_run(run_name='XGBoost_HyperparamSearch'):
    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=50, show_progress_bar=True)
    best_params = study.best_params
    best_score  = study.best_value
    print(f'Best Val AUC: {best_score:.5f}')
    mlflow.log_params({f'best_{k}': v for k, v in best_params.items()})
    mlflow.log_metric('best_val_auc', best_score)

  0%|          | 0/50 [00:00<?, ?it/s]

Best Val AUC: 0.97654
🏃 View run XGBoost_HyperparamSearch at: https://dagshub.com/giomamaca/ML2.mlflow/#/experiments/0/runs/060d21e9642041e68e14134ddae3fc36
🧪 View experiment at: https://dagshub.com/giomamaca/ML2.mlflow/#/experiments/0


In [25]:
with mlflow.start_run(run_name='XGBoost_Final_Model'):
    FINAL_PARAMS = {
        **best_params,
        'scale_pos_weight'     : (y==0).sum()/(y==1).sum(),
        'eval_metric'          : 'auc',
        'early_stopping_rounds': 50,
        'random_state'         : SEED,
        'n_jobs'               : -1,
        'tree_method'          : 'hist'
    }

    FINAL_PARAMS_FULL = {k: v for k, v in FINAL_PARAMS.items()
                         if k != 'early_stopping_rounds'}

    oof = np.zeros(len(y))
    for fold, (tr_idx, val_idx) in enumerate(SKF.split(X_sel, y), 1):
        m = xgb.XGBClassifier(**FINAL_PARAMS)
        m.fit(X_sel.iloc[tr_idx], y.iloc[tr_idx],
              eval_set=[(X_sel.iloc[val_idx], y.iloc[val_idx])],
              verbose=False)
        oof[val_idx] = m.predict_proba(X_sel.iloc[val_idx])[:, 1]

    final_auc = roc_auc_score(y, oof)
    print(f'Final OOF AUC: {final_auc:.5f}')

    preprocessor.feature_names_ = SELECTED_FEATURES

    final_clf = xgb.XGBClassifier(**FINAL_PARAMS_FULL)
    final_clf.fit(X_sel, y)

    fraud_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier',   final_clf)
    ])

    mlflow.log_metric('final_oof_auc', final_auc)
    mlflow.sklearn.log_model(
        fraud_pipeline,
        artifact_path='xgboost_pipeline',
        registered_model_name='XGBoost_Fraud_Pipeline'
    )

Final OOF AUC: 0.97482


2026/05/05 16:27:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/05 16:27:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'XGBoost_Fraud_Pipeline'.
2026/05/05 16:27:49 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost_Fraud_Pipeline, version 1
Created version '1' of model 'XGBoost_Fraud_Pipeline'.


🏃 View run XGBoost_Final_Model at: https://dagshub.com/giomamaca/ML2.mlflow/#/experiments/0/runs/e2ec3b6ba7fd43fe9c6049476c91247f
🧪 View experiment at: https://dagshub.com/giomamaca/ML2.mlflow/#/experiments/0
